In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, "../src")
import numpy as np
from energy import load_framework, load_force_field, pairwise_lj_params
from widom import widom, rho_framework, henry_coefficient


In [2]:
positions, symbols, L, masses = load_framework("../data/IRMOF-1.cif")
lj_params = load_force_field("../data/force_field/generic_dreiding_uff_mixing_rules.def")
eps_pair, sigma_pair = pairwise_lj_params(symbols, lj_params, guest=(158.5, 3.72))
rho_fw = rho_framework(masses, L)
print(f"rho_fw = {rho_fw:.1f} kg/m^3 (literature approx. 590)")


rho_fw = 593.3 kg/m^3 (literature approx. 590)


/Users/kasperhuysentruyt/Documents/leven /Cusp/cusp-mlff-gcmc/.venv/lib/python3.14/site-packages/ase/io/cif.py:411: UserWarning: crystal system 'cubic' is not interpreted for space group Spacegroup(225, setting=1). This may result in wrong setting!
  warnings.warn(


In [3]:
# Widom: N insertions, error via 10 block averages. Error decreases as sqrt(N); 10,000 insertions ~ 0.4 s.
N = 100_000
T = 298.0  # K
rng = np.random.default_rng(seed=100)

boltzmann_mean, sem = widom(N, T, positions, eps_pair, sigma_pair, L, rng, n_blocks=10)
K_H = henry_coefficient(boltzmann_mean, T, rho_fw)
K_H_err = henry_coefficient(sem, T, rho_fw)  # same denominator
print(f"<exp(-U/T)> = {boltzmann_mean:.3f} ± {sem:.3f}")
print(f"K_H(298 K) = {K_H:.3e} ± {K_H_err:.1e} mol/kg/Pa")


<exp(-U/T)> = 5.634 ± 0.027
K_H(298 K) = 3.832e-06 ± 1.8e-08 mol/kg/Pa


In [4]:
p = 2*10**5 # 1bar in Pascal 
mu_2_kg = 1.66053906892*10**(-27) #kg
avogadro = 6.02214076*10**(23) #mol^-1
mol = K_H * p* np.sum(masses)*mu_2_kg*avogadro  
print(mol) #Ch4 per cell 


4.7206027344077315


## Comparison with literature

No published K_H for CH4 in IRMOF-1 with exactly this force field (DREIDING/UFF, CH4 eps 158.5 K / sigma 3.72 A, 12 A shifted cutoff) was found. The closest low-loading reference is Düren et al., *Langmuir* 2004, 20, 2683 (Table 1): Q_st = 10.6 kJ/mol for CH4 in IRMOF-1 at 298 K with DREIDING and CH4 eps 148 K.

K_H is therefore checked indirectly in notebook 04: the GCMC loading at 1 bar agrees within 2% with K_H · p (2.36 CH4 per cell), and the isotherm at 35 bar lies ~13% above Düren, which is explained by the 7% deeper CH4 well (158.5 K versus 148 K).